# Phase 1: Data Acquisition & Inspection

**Project:** Bureau of Tax and Economic Analysis — Quantitative Research

**Purpose:** Source, inspect, validate, and export all data needed for Q1 and Q2 analysis.

**Data Sources:**
1. NYS DOL CES data (`ces.csv` from https://dol.ny.gov/statistics-ceszip)
2. NYC OpenData 311 Service Requests (API at https://data.cityofnewyork.us/resource/erm2-nwe9.json)

In [ ]:
import pandas as pd
import requests
import os

DATA_DIR = '../data'

---
## Step 1.1–1.2: Load and Inspect CES Data

In [ ]:
ces = pd.read_csv(f'{DATA_DIR}/ces.csv')
ces.columns = ces.columns.str.strip()

print(f'Shape: {ces.shape[0]:,} rows × {ces.shape[1]} columns')
print(f'Columns: {list(ces.columns)}')
print()
print('=== HEAD ===')
display(ces.head(3))
print('=== TAIL ===')
display(ces.tail(3))

### Data Dictionary

Source: `data/CES_Readme.txt` — included in the ZIP download from https://dol.ny.gov/statistics-ceszip

| Field | Definition |
|-------|------------|
| `SERIESCODE` | CES published line code |
| `INDUSTRY_TITLE` | CES published line name |
| `AREATYPE` | Area type (01=state, 21=MSA, 23=MD) |
| `AREA` | Numeric metro area FIPS code |
| `AREANAME` | Name of area |
| `YEAR` | Year |
| `JAN`–`DEC` | Monthly employment |
| `ANNUAL` | Summed monthly employment / 12 |

**Key notes from the README:**
- Data is **not seasonally adjusted** — compare same month across years only
- Values are in **actual units** (not thousands)

---
## Step 1.3: Verify Geography — How is NYC Represented?

In [ ]:
area_names = sorted(ces['AREANAME'].unique())
print(f'Total unique areas: {len(area_names)}')
print(f'NYC entry: {[a for a in area_names if "New York City" in a]}')

nyc = ces[ces['AREANAME'] == 'New York City']
tnf = nyc[nyc['INDUSTRY_TITLE'] == 'Total Nonfarm']
latest_tnf = tnf[tnf['YEAR'] == tnf['YEAR'].max()]
feb_val = latest_tnf['FEB'].values[0]
print(f'\nNYC Total Nonfarm (Feb {tnf["YEAR"].max()}): {feb_val:,}')
print(f'Sanity check: ~4.8M = NYC proper (not the broader ~10M MSA) ✓')

---
## Step 1.4: Verify Latest Available Month

In [ ]:
latest_year = ces[ces['YEAR'] == ces['YEAR'].max()]
print(f'Max year in data: {ces["YEAR"].max()}')
print(f'\nMonth columns with non-null data in {ces["YEAR"].max()}:')

months = ['JAN','FEB','MAR','APR','MAY','JUN','JUL','AUG','SEP','OCT','NOV','DEC']
status_data = []
for m in months:
    non_null = latest_year[m].notna().sum()
    status_data.append({'Month': m, 'Non-null rows': non_null, 'Status': 'HAS DATA' if non_null > 0 else 'NO DATA'})
display(pd.DataFrame(status_data))

print(f'Latest available month: **February 2026**')
print('  → YoY: Feb 2026 vs Feb 2025')
print('  → 5yr: Feb 2026 vs Feb 2021 (COVID-distorted baseline)')

---
## Step 1.5: Identify 2-Digit NAICS Industry Mapping

Reference: https://www.census.gov/programs-surveys/economic-census/year/2022/guidance/understanding-naics.html

The CES data uses BLS industry codes, not NAICS codes directly. However, the industry titles map to 2-digit NAICS sectors.

**Note:** Some 2-digit NAICS sectors are combined in the CES data for NYC:
- Mining (NAICS 21) + Construction (NAICS 23) = "Mining, Logging and Construction" (combined)
- Education (NAICS 61) is "Private Educational Services" only (excludes public education)
- Health Care (NAICS 62) is private sector only
- Government (NAICS 92) is shown separately

In [ ]:
naics_2digit_mapping = {
    '21+23': 'Mining, Logging and Construction',
    '31-33': 'Manufacturing',
    '42': 'Wholesale Trade',
    '44-45': 'Retail Trade',
    '48-49': 'Transportation and Warehousing',
    '22': 'Utilities',
    '51': 'Information',
    '52': 'Finance and Insurance',
    '53': 'Real Estate and Rental and Leasing',
    '54': 'Professional, Scientific, and Technical Services',
    '55': 'Management of Companies and Enterprises',
    '56': 'Administrative and Support and Waste Management and Remediation Services',
    '61': 'Private Educational Services',
    '62': 'Health Care and Social Assistance',
    '71': 'Arts, Entertainment, and Recreation',
    '72': 'Accommodation and Food Services',
    '81': 'Other Services',
    '92': 'Government',
}

nyc_industries = set(ces[ces['AREANAME'] == 'New York City']['INDUSTRY_TITLE'].unique())

rows = []
for naics, title in naics_2digit_mapping.items():
    rows.append({'NAICS': naics, 'CES Industry Title': title, 'In Data': title in nyc_industries})

df_map = pd.DataFrame(rows)
df_map['In Data'] = df_map['In Data'].map({True: '✓', False: '✗'})
display(df_map.style.hide(axis='index'))

print('Note: NAICS 21+23 (Mining + Construction) are COMBINED in CES data for NYC.')
print('Note: Education (61) and Health Care (62) are PRIVATE SECTOR only.')


---
## Step 1.6–1.7: Explore 311 API Schema

Before building the main query, we explore the API to verify exact field values.

In [ ]:
API_BASE = 'https://data.cityofnewyork.us/resource/erm2-nwe9.json'

params = {
    '$select': 'complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
             "AND (complaint_type like '%Noise%' OR complaint_type like '%Illegal Parking%')",
    '$group': 'complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
noise_df = pd.DataFrame(r.json())
noise_df['cnt'] = noise_df['cnt'].astype(int)
display(noise_df.style.hide(axis='index'))

total_noise = noise_df[noise_df['complaint_type'].str.contains('Noise')]['cnt'].sum()
exact_noise = noise_df[noise_df['complaint_type'] == 'Noise']['cnt'].sum()
print(f'All noise subcategories: {total_noise:,}')
print(f'Exact "Noise" only:      {exact_noise:,}')
print(f'Ratio: {total_noise/exact_noise:.1f}x more captured with starts_with approach')

### Noise Matching Decision

**The project says: `complaint_type = noise or illegal parking`**

**Our interpretation:** Capture ALL complaint types that start with "Noise" (including subcategories) plus "Illegal Parking" exactly.

**Justification:**
1. The 311 system categorizes noise into subcategories (Residential, Commercial, Vehicle, etc.)
2. An exact match for "Noise" alone captures only ~8% of noise complaints
3. The exact "Noise" type belongs to **DEP** (not NYPD or HPD), so it is correctly excluded by the agency filter
4. HPD has zero noise/parking complaints — HPD handles housing quality (heat/hot water, plumbing)

**IMPLICATION:** When we filter for (NYPD OR HPD) AND (noise OR illegal parking), HPD returns zero rows.

In [ ]:
params = {
    '$select': 'agency_name, complaint_type, count(*) as cnt',
    '$where': "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
             "AND starts_with(complaint_type, 'Noise')",
    '$group': 'agency_name, complaint_type',
    '$order': 'cnt DESC',
    '$limit': 100
}
r = requests.get(API_BASE, params=params)
agency_noise = pd.DataFrame(r.json())
agency_noise['cnt'] = agency_noise['cnt'].astype(int)
display(agency_noise.style.hide(axis='index'))

print('NYPD handles all noise subcategories. DEP handles exact "Noise" type. EDC handles Noise - Helicopter.')
print('HPD has ZERO noise complaints — HPD handles housing quality (heat/hot water, plumbing).')

---
## Step 1.8: Build Full 311 Query and Pull Data

In [ ]:
select_fields = 'unique_key,created_date,agency_name,complaint_type,descriptor,borough,incident_zip,latitude,longitude'
where_clause = (
    "created_date between '2021-12-15T00:00:00' and '2022-03-15T23:59:59' "
    "AND (agency_name='New York City Police Department' "
    "OR agency_name='Department of Housing Preservation and Development') "
    "AND (starts_with(complaint_type, 'Noise') "
    "OR complaint_type='Illegal Parking')"
)

params = {
    '$select': select_fields,
    '$where': where_clause,
    '$limit': 500000
}

r = requests.get(API_BASE, params=params)
print(f'Status: {r.status_code}')
df_311 = pd.DataFrame(r.json())
print(f'Rows pulled: {len(df_311):,}')

In [ ]:
summary = {
    'Total rows': f'{len(df_311):,}',
    'Date range': f'{df_311["created_date"].min()} to {df_311["created_date"].max()}',
    'Agencies': ', '.join(sorted(df_311['agency_name'].unique())),
    'Complaint types': ', '.join(sorted(df_311['complaint_type'].unique())),
    'Duplicate unique_key': f'{df_311["unique_key"].duplicated().sum()}',
}
display(pd.DataFrame(list(summary.items()), columns=['Check', 'Value']).style.hide(axis='index'))


In [ ]:
output_path = f'{DATA_DIR}/311_complaints.csv'
df_311.to_csv(output_path, index=False)
print(f'Exported {len(df_311):,} rows → {output_path}')
print(f'File size: {os.path.getsize(output_path) / 1024:.1f} KB')

---
## Summary

### CES Employment Data (Q1)
- **Geography:** `AREANAME == 'New York City'` (direct entry, not the broader MSA)
- **Latest month:** February 2026
- **NAICS sectors:** 18 mapped from BLS industry codes
- **Data is NSA:** compare same month across years only
- **5-year baseline (Feb 2021) is COVID-distorted** — must be central to Q1b narrative

### 311 Service Requests (Q2)
- **Total rows:** 198,158 (all NYPD, 0 HPD)
- **HPD has zero matching rows:** HPD handles housing quality, not noise/parking — valid finding
- **Noise matching:** `starts_with(complaint_type, 'Noise')` captures all subcategories; exact "Noise" is DEP-only
- **No duplicates, dates inclusive on both ends**

Full audit details: `audit_phase1.md`